## 🎓 Parte 01 — Populações, Amostras & Distribuições (Aulas 1–4)

<br>

<div align="center">

| Aula | Título (direto) | Foco Principal |
|:---:|:-----------------|:---------------|
| 1 | Populações e Amostras | Intuição + estimadores básicos |
| 2 | Viés | Falhas de representatividade e como evitá-las |
| 3 | Tipos de Amostragem | SRS, Estratificada, Sistemática (trade-offs) |
| 4 | Distribuições Amostrais | Variabilidade de estatísticas e por que importa |

</div>

<br>


---
## 🎓 Aula 1: Populações e Amostras

<br>

### **Parte 1: Introdução e Conceito**

<br>

* **O Grande "Porquê":** no trabalho real, não medimos todo o universo — medimos *pedaços* bem escolhidos. A qualidade das decisões (preço, SLA, UX) depende de entender como a **amostra** fala sobre a **população**.
* **Fundamentos Chave:**
  * **População (\(N\))**: conjunto completo (ex.: todos os pedidos do iFood no mês).
  * **Amostra (\(n\))**: subconjunto selecionado para inferir sobre o todo.
  * **Média amostral \(\bar{X}\)**: estimativa de \(\mu\) (média populacional).
  * **Erro-padrão da média (EP):**  
    $$
    EP(\bar{X}) = \frac{\sigma}{\sqrt{n}} \;\approx\; \frac{s}{\sqrt{n}}
    $$
    **Parâmetros:** \(\sigma\) (desvio-pop, unidade do KPI), \(s\) (desvio amostral, \(ddof=1\)), \(n\ge 2\).

<br>


### **Parte 2: Prática e Aplicação (Projeto “Operação Entrega Relâmpago”)**

<br>

**Visão de Negócio:** estimar o **tempo médio de entrega** (KPI: `delivery_time_min`) com precisão suficiente para negociar SLA com parceiros, sem precisar varrer todos os pedidos.

**Plano:** (1) sortear amostra; (2) calcular \(\bar{X}\) e \(EP\); (3) comunicar intervalo de confiança ao time.

<br>


In [ ]:
# Micro-bloco 1 — RNG, índices e média (didactic placeholder)
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  # reproducible random generator
idx = rng.integers(0, 100_000, size=2_000, endpoint=False)  # sample indices (with replacement)
xbar = np.array([32.1, 28.7, 41.0])[1:].mean()  # simple placeholder to show .mean(); real data comes next


In [ ]:
# Pratique seu código aqui!


<br>

---
#### Linha 1: `rng = np.random.default_rng(42)`

<br>

- **🎯 Intenção:** criar gerador pseudoaleatório **reprodutível**.
- **⚙️ Objeto/Parâmetros:** `np.random.default_rng(42)` → instância `Generator` (PCG64). `42` é a *seed*.
- **📤 Saída:** `rng` (estado de aleatoriedade controlado).
- **⏱️ Custo:** O(1).
- **🧪 Mini-experimentos:**
  - Troque `42` por `0` → sequência muda.
  - Sem seed → resultados variam a cada execução.
  - Reutilize `rng` em todo experimento → garante replicabilidade.
- **🚦 Armadilhas:** esquecer seed atrapalha auditorias.
- **💼 KPI:** reprodutibilidade = confiança nas estimativas ao negociar SLA.
- **Em resumo:** definimos a fonte de aleatoriedade sob controle.

<br>

---
#### Linha 2: `idx = rng.integers(0, 100_000, size=2_000, endpoint=False)`

<br>

- **🎯 Intenção:** sortear `n=2000` **índices válidos** (SRS com reposição) de uma população com ~100k linhas.
- **🧮 Precisão da estimativa:**
  $$
  EP(\bar{X})\approx\frac{s}{\sqrt{n}}
  $$
  **Parâmetros:** \(s\) (desvio da métrica), \(n\) (tamanho da amostra).
- **⚙️ Método/Parâmetros:** `.integers(low=0, high=100_000, size=2_000, endpoint=False)`  
  `low` inclusivo; `high` exclusivo; `size` = quantidade; `endpoint=False` evita índice inválido.
- **📤 Saída:** `idx` é `ndarray(int)`, shape `(2000,)`. Pode repetir índices (com reposição).
- **⏱️ Custo:** O(n).
- **🧪 Mini-experimentos:**
  - Sem reposição: `np.random.choice(high, size=2000, replace=False)` → variância ligeiramente menor.
  - Quadruplicar `n` → EP cai ~2× (lei \(1/\sqrt{n}\)).
  - População com cauda longa → exija `n` maior para estabilidade.
- **🚦 Armadilhas:** confundir `endpoint=True` (não usar para indexar).
- **💼 KPI:** base para estimar média de entrega com custo fixo de coleta.
- **Em resumo:** criamos os ponteiros da amostra.

<br>

---
#### Linha 3: `xbar = np.array([32.1, 28.7, 41.0])[1:].mean()`

<br>

- **🎯 Intenção:** *placeholder* apenas para ilustrar `.mean()` (usaremos dados reais no próximo bloco).
- **🧮 Fórmula da média:**
  $$
  \bar{x} = \frac{1}{n}\sum_{i=1}^n x_i
  $$
- **⚙️ Método:** `.mean()` calcula a média; sem `axis` em 1D retorna escalar.
- **🧪 Mini-experimentos:** `dtype=np.float64`; `axis=0/1` em matrizes; `where` para média condicional.
- **🚦 Armadilhas:** NaNs → use `np.nanmean`.
- **Em resumo:** média resume a amostra em um número interpretável.

<br>

---
### Parâmetros Adicionais (Não Utilizados no Código)

<br>

- `rng.integers(..., dtype=np.int32)` — economiza memória de índices.

<br>


In [ ]:
# Example: int32 indices (same logic, smaller footprint)
idx = rng.integers(0, 100_000, size=2_000, dtype=np.int32, endpoint=False)


In [ ]:
# Pratique seu código aqui!


- `.mean(axis=0, dtype=np.float64, keepdims=True, where=mask)` — médias por coluna, com precisão e máscara.

<br>


In [ ]:
# Example: mean with precision, keeping dims and using a mask
arr = np.array([10.0, 0.0, 20.0, np.nan])
mask = ~np.isnan(arr) & (arr > 0)
xbar_masked = arr.mean(dtype=np.float64, keepdims=False, where=mask)


In [ ]:
# Pratique seu código aqui!


<br>


In [ ]:
# Micro-bloco 2 — Carregar dados reais (iFood) ou simular (fallback)
import pandas as pd, numpy as np

try:
    df = pd.read_csv('/mnt/data/base_ifood_limpa.csv')  # adjust path in Colab if needed
    population = df['delivery_time_min'].dropna().to_numpy()
except Exception:
    rng = np.random.default_rng(123)
    population = rng.gamma(shape=4.0, scale=8.0, size=120_000)  # realistic right-skewed times

rng = np.random.default_rng(7)
n = 2000
idx = rng.integers(0, len(population), size=n, endpoint=False)
sample = population[idx]
xbar = sample.mean()
s = sample.std(ddof=1)
ep = s/np.sqrt(n)


In [ ]:
# Pratique seu código aqui!


<br>

---
#### Linhas finais do Micro-bloco 2

<br>

- **🎯 Intenção:** obter `sample`, calcular \(\bar{x}\), \(s\) (com \(ddof=1\)) e \(EP=s/\sqrt{n}\).
- **🧮 Fórmulas:**
  $$
  \bar{x}=\frac{1}{n}\sum x_i,\quad s=\sqrt{\frac{1}{n-1}\sum(x_i-\bar{x})^2},\quad EP=\frac{s}{\sqrt{n}}
  $$
- **🚦 Armadilhas:** `ddof=0` subestima variância em \(n\) pequeno; NaNs contaminam \(\bar{x}\) → `np.nanmean`.
- **💼 KPI:** traduz EP em **± minutos** de incerteza no SLA.

<br>


### **📝 Nota de Rodapé para Novatos**

<br>

* **EP (Erro-Padrão):** dispersão da *média amostral*; diminui com \(n\).
* **`ddof=1`:** correção de Bessel — divide por \(n-1\) para estimador não viesado.
* **Reprodutibilidade:** “seed” fixa permite repetir resultados.

<br>


### **Parte 3: Verificação de Aprendizado (Aula 1)**

<br>

1. A média amostral \(\bar{X}\) estima:
* A) \(\sigma\)
* B) \(\mu\)
* C) \(s\)
* D) \(n\)
* E) \(\alpha\)

2. O erro-padrão da média cai com:
* A) \(1/n\)
* B) \(1/\sqrt{n}\)
* C) \(n\)
* D) \(n^2\)
* E) \(z\)

3. `ddof=1` é usado para:
* A) acelerar o cálculo
* B) reduzir EP
* C) tornar \(s\) não viesado
* D) normalizar dados
* E) incluir NaNs

4. Em SRS com reposição, índices podem:
* A) sair negativos
* B) repetir
* C) extrapolar `high`
* D) ser sempre únicos
* E) ser floats

5. Se a população tem caudas pesadas, para estabilizar \(\bar{X}\):
* A) reduzir \(n\)
* B) aumentar \(n\)
* C) usar \(ddof=0\)
* D) tirar a mediana
* E) trocar unidade

<br>

### **🔑 Gabarito (Aula 1)**

<br>

1. **B** — \(\bar{X}\) é estimador de \(\mu\).  
2. **B** — \(EP \propto 1/\sqrt{n}\).  
3. **C** — correção de Bessel.  
4. **B** — com reposição, pode repetir.  
5. **B** — mais \(n\) reduz variância da média.

<br>


---
## 🎓 Aula 2: Viés

<br>

### **Parte 1: Introdução e Conceito**

<br>

* **O Grande "Porquê":** estimativas boas com **amostras ruins** enganam. Viés é erro **sistemático** que desloca a inferência.
* **Principais vieses (exemplos práticos):**
  * **Auto-seleção:** só clientes muito satisfeitos respondem NPS.
  * **Cobertura:** app novo testado só em 4G urbano.
  * **Não-resposta:** segmentos silenciosos somem da amostra.

**Definição (viés de um estimador \(\hat{\theta}\)):**
$$
\text{Viés}(\hat{\theta})=\mathbb{E}[\hat{\theta}]-\theta
$$
**Parâmetros:** \(\theta\) (parâmetro real), \(\hat{\theta}\) (estimador), \(\mathbb{E}[\cdot]\) (média teórica).

<br>


### **Parte 2: Prática e Aplicação (Auditoria de Amostra)**

<br>

**Visão de Negócio:** checar se a amostra representa a população **por cidade** (orçamento de marketing regional).

**Plano:** (1) proporções por cidade na população vs amostra; (2) gaps; (3) flag de risco.

<br>


In [ ]:
# Micro-bloco 1 — Proporções por cidade e gaps
pop_city = df["city"].value_counts(normalize=True).sort_index()
smp_city = df.iloc[idx]["city"].value_counts(normalize=True).sort_index()

audit = pd.concat([pop_city.rename("pop"), smp_city.rename("smp")], axis=1).fillna(0.0)
audit["gap_pp"] = (audit["smp"] - audit["pop"])*100  # percentage points


In [ ]:
# Pratique seu código aqui!


<br>

---
#### Linha 1: `pop_city = df["city"].value_counts(normalize=True).sort_index()`

<br>

- **🎯 Intenção:** obter **proporções** por cidade para a população.
- **🧮 Fórmula:** para cada cidade \(h\),
  $$
  \hat{p}_h=\frac{\text{contagem}_h}{\sum_k \text{contagem}_k}
  $$
- **⚙️ Método/Parâmetros:** `value_counts(normalize=True)` → frequência relativa; `sort_index()` alinha por rótulo.
- **📤 Saída:** `Series` com \(\sum \hat{p}_h = 1\).
- **🧪 Mini-experimentos:** `dropna=False`; `sort=False` para preservar ordem original.
- **🚦 Armadilhas:** comparar sem mesmo índice → *reindex* antes.
- **💼 KPI:** base para metas proporcionais por praça.
- **Em resumo:** referência populacional por cidade.

<br>

---
#### Linha 2: `smp_city = df.iloc[idx]["city"].value_counts(normalize=True).sort_index()`

<br>

- **🎯 Intenção:** mesmas proporções **na amostra**.
- **📤 Saída:** `Series` alinhável a `pop_city`.
- **🧪 Mini-experimentos:** `idx` sem reposição → menos variação.
- **Em resumo:** espelho amostral para comparação.

<br>

---
#### Linha 3–4: concat, `fillna(0.0)`, gap em pp

<br>

- **🎯 Intenção:** juntar e medir **desvios** (amostra − população).
- **🧮 Gap (pp):**
  $$
  \text{gap}_h = (\hat{p}_h^{smp}-\hat{p}_h^{pop})\times 100
  $$
- **📤 Saída:** DataFrame `audit` com colunas `pop`, `smp`, `gap_pp`.
- **🧪 Mini-experimentos:** thresholds (`abs(gap_pp)>2pp`) como alerta.
- **🚦 Armadilhas:** estratos raros → variância alta do gap; use bandas.
- **💼 KPI:** risco de **sub/sobreexposição** regional.
- **Em resumo:** diagnóstico objetivo de representatividade.

<br>

---
### Parâmetros Adicionais (Não Utilizados no Código)

<br>

- `value_counts(normalize=True, dropna=False, ascending=True)` — inclua faltantes; inverta ordem.

<br>


In [ ]:
# Example: include NaNs and invert order
pop_city_alt = df["city"].value_counts(normalize=True, dropna=False, ascending=True)


In [ ]:
# Pratique seu código aqui!


- `pd.concat(..., join='outer', keys=[...])` — auditar múltiplos recortes (cidade×canal).

<br>


In [ ]:
# Example: combine multiple audits with keys
audit_multi = pd.concat(
    {"city": pop_city, "channel": df["channel"].value_counts(normalize=True)},
    axis=1, join="outer"
)


In [ ]:
# Pratique seu código aqui!


<br>


### **Parte 3: Verificação de Aprendizado (Aula 2)**

<br>

1. Viés é:
* A) ruído aleatório
* B) erro sistemático
* C) erro numérico
* D) overfitting
* E) subamostragem

2. `dropna=False` em `value_counts` serve para:
* A) acelerar
* B) incluir faltantes como categoria
* C) excluir duplicados
* D) normalizar por log
* E) remover outliers

3. Gap em pp é adequado para comparar:
* A) médias
* B) desvios
* C) proporções
* D) quantis
* E) variâncias

4. Estratos raros têm:
* A) variância menor
* B) variância maior
* C) mesmo EP
* D) EP=0
* E) gap sempre 0

5. Representatividade prática costuma usar:
* A) teste exato de Fisher
* B) banda de tolerância em pp
* C) regressão linear
* D) PCA
* E) k-means

<br>

### **🔑 Gabarito (Aula 2)**

<br>

1. **B** — desloca a estimativa.  
2. **B** — inclui NaNs como categoria.  
3. **C** — diferenças de percentuais.  
4. **B** — amostras pequenas → maior variância.  
5. **B** — critério operacional simples.

<br>


---
## 🎓 Aula 3: Tipos de Amostragem

<br>

### **Parte 1: Introdução e Conceito**

<br>

* **O Grande "Porquê":** quando a população é **heterogênea**, SRS pode falhar; escolher o tipo certo reduz viés/variância.
* **Resumo rápido:**
  * **SRS (Simples):** fácil; pode ignorar subgrupos raros.
  * **Estratificada:** amostra por estrato \(h\) com alocação \(n_h\).  
    **Proporcional:** \(n_h = n \cdot \frac{N_h}{N}\).  
    **Ótima (Neyman):** \(n_h \propto N_h\sigma_h\).
  * **Sistemática:** 1 em cada \(k\) após arranjo aleatório (cuidado com periodicidade).

<br>


### **Parte 2: Prática e Aplicação (Estratificada por cidade)**

<br>

**Visão de Negócio:** garantir representação mínima de cidades pequenas no KPI de entrega.

**Plano:** (1) calcular pesos \(N_h/N\); (2) alocar \(n_h\); (3) amostrar por estrato.

<br>


In [ ]:
# Micro-bloco 1 — Alocação proporcional e amostragem estratificada
counts = df["city"].value_counts().sort_index()
weights = counts / counts.sum()                # P_h = N_h/N
n = 2000
nh = (weights * n).round().astype(int).clip(lower=1)

smp_idx = []
rng = np.random.default_rng(12)
for city, take in nh.items():
    pool = df.index[df["city"].eq(city)].to_numpy()
    take = min(take, len(pool))
    smp_idx.append(rng.choice(pool, size=take, replace=False))
smp_idx = np.concatenate(smp_idx)
smp = df.loc[smp_idx, "delivery_time_min"].dropna().to_numpy()


In [ ]:
# Pratique seu código aqui!


<br>

---
#### Linha 1–3: `counts`, `weights`, `nh`

<br>

- **🎯 Intenção:** calcular \(P_h=N_h/N\) e alocar \(n_h=n\cdot P_h\).
- **🧮 Fórmulas:**
  $$
  P_h=\frac{N_h}{N}, \quad n_h^\text{prop}=n\cdot P_h
  $$
- **📤 Saída:** `nh` inteiro, mínimo 1 por estrato.
- **🧪 Mini-experimentos:** alocação de Neyman \(n_h \propto N_h\sigma_h\) reduz EP quando \(\sigma_h\) difere.

<br>

---
#### Linha 4–9: laço de amostragem por cidade

<br>

- **🎯 Intenção:** amostrar **sem reposição** dentro de cada estrato.
- **⚙️ Método:** `rng.choice(pool, size=take, replace=False)` garante unicidade.
- **📤 Saída:** `smp_idx` concatenado; `smp` vetor de tempos.
- **🧪 Mini-experimentos:** impor **cap** de máximo por cidade para custo.
- **🚦 Armadilhas:** estratos pequenos → `take>len(pool)`; guarde `min`.
- **💼 KPI:** estabilidade do KPI por praça (comparabilidade regional).
- **Em resumo:** garantimos cobertura de todos os estratos.

<br>

---
### Parâmetros Adicionais (Não Utilizados no Código)

<br>

- Alocação ótima (Neyman): usar pesos por variabilidade \(\sigma_h\).

<br>


In [ ]:
# Example: Neyman allocation (requires per-stratum std, sigma_by_city)
sigma_by_city = df.groupby("city")["delivery_time_min"].std(ddof=1)
w_neyman = counts * sigma_by_city
nh_neyman = (n * w_neyman / w_neyman.sum()).round().astype(int).clip(lower=1)


In [ ]:
# Pratique seu código aqui!


- Amostragem sistemática (cuidado com periodicidade).

<br>


In [ ]:
# Example: systematic sampling over a shuffled frame
rng = np.random.default_rng(33)
shuffled = df.sample(frac=1.0, random_state=33).reset_index(drop=True)
k = int(len(shuffled)/n)
start = rng.integers(0, k)
sys_idx = np.arange(start, start + k*n, k)
sys_sample = shuffled.loc[sys_idx, "delivery_time_min"].to_numpy()


In [ ]:
# Pratique seu código aqui!


<br>


### **Parte 3: Verificação de Aprendizado (Aula 3)**

<br>

1. Em população heterogênea, estratificada tende a:
* A) aumentar EP
* B) reduzir EP
* C) não mudar EP
* D) inviabilizar IC
* E) sempre enviesar

2. Alocação proporcional define \(n_h\) como:
* A) \(n/N_h\)
* B) \(n\cdot N_h/N\)
* C) \(N_h/n\)
* D) \(n/N\)
* E) \(N/n_h\)

3. Na sistemática, risco principal:
* A) não-resposta
* B) periodicidade da lista
* C) overfitting
* D) underflow
* E) normalização

4. No estrato muito pequeno, devemos:
* A) permitir `replace=True`
* B) `min(take, len(pool))`
* C) ignorar o estrato
* D) duplicar registros
* E) usar `endpoint=True`

5. Neyman é melhor quando:
* A) \(\sigma_h\) idênticos
* B) \(\sigma_h\) diferentes
* C) \(N_h\) iguais
* D) \(n\) é 1
* E) dados binários sempre

<br>

### **🔑 Gabarito (Aula 3)**

<br>

1. **B** — homogeneiza dentro dos estratos.  
2. **B** — proporcional ao peso do estrato.  
3. **B** — pode “pegar” padrão periódico.  
4. **B** — protege contra estouro.  
5. **B** — aloca mais onde a variância é maior.

<br>


---
## 🎓 Aula 4: Distribuições Amostrais

<br>

### **Parte 1: Introdução e Conceito**

<br>

* **O Grande "Porquê":** repetir o processo de amostragem gera uma **distribuição** de \(\bar{X}\). Isso quantifica a **variabilidade** do nosso estimador.
* **Definições:**
  * **Distribuição amostral de \(\bar{X}\):** distribuição das médias sobre muitas amostras.
  * **EP de \(\bar{X}\):** desvio dessa distribuição.
* **Fórmulas-chave:**
  $$
  \mathbb{E}[\bar{X}]=\mu,\qquad \operatorname{Var}(\bar{X})=\frac{\sigma^2}{n}
  $$

<br>


### **Parte 2: Prática e Aplicação (Simulação de \(\bar{X}\))**

<br>

**Visão de Negócio:** estimar a **estabilidade** do KPI (média de entrega) ao variar \(n\).

**Plano:** (1) simular muitas amostras; (2) coletar \(\bar{X}\); (3) comparar EP empírico vs. teórico.

<br>


In [ ]:
# Micro-bloco 1 — Simular distribuição amostral de Xbar
import numpy as np

rng = np.random.default_rng(101)
population = rng.gamma(shape=4.0, scale=8.0, size=200_000)  # realistic skewed population
n = 400
B = 2000

xbars = np.empty(B, dtype=float)
for b in range(B):
    idx = rng.integers(0, len(population), size=n, endpoint=False)
    xbars[b] = population[idx].mean()

ep_emp = xbars.std(ddof=1)           # empirical SE
s_pop  = population.std(ddof=1)      # approx σ
ep_the = s_pop/np.sqrt(n)            # theoretical SE


In [ ]:
# Pratique seu código aqui!


<br>

---
#### Interpretação

<br>

- **🎯 Objetivo:** comparar `ep_emp` com `ep_the`.
- **🧮 Fórmulas:**
  $$
  EP_\text{emp}=\operatorname{sd}(\{\bar{x}_b\}),\quad EP_\text{the} \approx \frac{s_{pop}}{\sqrt{n}}
  $$
- **Leitura:** valores próximos validam a teoria mesmo com população não normal (prévia do TLC).
- **🚦 Armadilhas:** \(B\) pequeno → `ep_emp` ruidoso; amostragem sem reposição requer **correção finita**.
- **💼 KPI:** quão “nervoso” fica o KPI com amostras deste tamanho.

<br>

---
### Parâmetros Adicionais (Não Utilizados no Código)

<br>

- Correção de população finita (sem reposição):
  $$
  EP(\bar{X})=\frac{\sigma}{\sqrt{n}}\sqrt{\frac{N-n}{N-1}}
  $$

<br>


In [ ]:
# Example: finite population correction (FPC) when sampling without replacement
N = 200_000
ep_fpc = (s_pop/np.sqrt(n)) * np.sqrt((N - n)/(N - 1))


In [ ]:
# Pratique seu código aqui!


- Paralelizar simulação (NumPy vectorize / Numba) para B grande.

<br>


In [ ]:
# Example: vectorized draw (no Python loop) for speed when feasible
idx_all = rng.integers(0, len(population), size=(B, n), endpoint=False)
xbars_vec = population[idx_all].mean(axis=1)


In [ ]:
# Pratique seu código aqui!


<br>


### **Parte 3: Verificação de Aprendizado (Aula 4)**

<br>

1. A distribuição de \(\bar{X}\) descreve:
* A) dados brutos
* B) média quando repetimos amostragens
* C) população
* D) resíduos
* E) variância

2. O EP empírico é:
* A) \(\bar{x}\)
* B) \(s\)
* C) desvio de `xbars`
* D) mediana de `xbars`
* E) variância amostral

3. A correção finita reduz EP quando:
* A) \(N\gg n\)
* B) \(n/N\) grande
* C) \(n=1\)
* D) \(B\to\infty\)
* E) \(\sigma=0\)

4. População assimétrica + \(n\) adequado:
* A) \(\bar{X}\) segue exatamente gamma
* B) \(\bar{X}\) aproxima normal
* C) \(\bar{X}\) é uniforme
* D) \(\bar{X}\) é Cauchy
* E) impossível estimar

5. Aumentar \(B\) muda:
* A) \(\mu\)
* B) a **precisão** da estimativa do EP empírico
* C) EP teórico
* D) unidade
* E) amostra original

<br>

### **🔑 Gabarito (Aula 4)**

<br>

1. **B** — estatística sob repetição de amostras.  
2. **C** — desvio de `xbars`.  
3. **B** — quando \(n\) é fração relevante de \(N\).  
4. **B** — prévia do TLC.  
5. **B** — estima EP com menor ruído (não muda EP teórico).

<br>


---
## 🔚 Resumo 15s (Parte 01)

<br>

- **Amostras boas contam a verdade do todo:** \(\bar{X}\) é simples, poderosa e precisa com \(EP\propto 1/\sqrt{n}\).
- **Viés dói mais que ruído:** audite representatividade (gaps em pp, estratos críticos).
- **Distribuições amostrais ≠ “mágica”:** elas só registram o quão “nervoso” seu estimador é; aumente \(n\) para estabilidade.

<br>


---
## 📌 Próximo passo sugerido

<br>

Se quiser, sigo para a **Parte 02 (Aulas 5–6 e 8)** no mesmo padrão “Ajustado”, conectando TLC ⇢ tamanho amostral ⇢ avaliação em Python (QQ/KS, pós-estratificação, pesos e caps).

<br>
